In [2]:
#import lib 
import os
import sys 
import tarfile
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from six.moves import urllib

In [3]:
sess = tf.Session()

In [4]:
# define model parameters
batch_size = 128
output_every = 50
generations = 20000
eval_every = 500
image_height = 32
image_width = 32
crop_height = 24
crop_width = 24
num_channels = 3
data_dir = 'datasets/'
extract_folder = 'cifar-10-batch-bin'

In [5]:
learning_rate = 0.1
lr_decay = 0.9
num_gens_to_wait = 250

In [6]:
image_vec_length = image_height*image_width*num_channels
record_length = 1 + image_vec_length

In [7]:
# Load data
data_dir = 'datasets'
if not os.path.exists(data_dir):
    os.makedirs(data_dir)
cifar10_url = 'http://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz'

# Check if file exists, otherwise download it
data_file = os.path.join(data_dir, 'cifar-10-binary.tar.gz')
if os.path.isfile(data_file):
    pass
else:
    # Download file
    def progress(block_num, block_size, total_size):
        progress_info = [cifar10_url, float(block_num * block_size) / float(total_size) * 100.0]
        print('\r Downloading {} - {:.2f}%'.format(*progress_info), end="")
    filepath, _ = urllib.request.urlretrieve(cifar10_url, data_file, progress)
    # Extract file
    tarfile.open(filepath, 'r:gz').extractall(data_dir)

In [8]:
#set up the record reader 
#return randomly distorted image
def read_cifar_files(filename_queue, distort_images = True):
    reader = tf.FixedLengthRecordReader(record_bytes=record_length)
    key, record_string = reader.read(filename_queue)
    record_bytes = tf.decode_raw(record_string, tf.uint8)
    image_label = tf.cast(tf.slice(record_bytes, [0], [1]), tf.int32)
  
    # Extract image
    image_extracted = tf.reshape(tf.slice(record_bytes, [1], [image_vec_length]),[num_channels, image_height, image_width])
    
    # Reshape image
    image_uint8image = tf.transpose(image_extracted, [1, 2, 0])
    reshaped_image = tf.cast(image_uint8image, tf.float32)
    # Randomly Crop image
    final_image = tf.image.resize_image_with_crop_or_pad(reshaped_image, crop_width, crop_height)
    
    if distort_images:
        # Randomly flip the image horizontally, change the brightness and contrast
        final_image = tf.image.random_flip_left_right(final_image)
        final_image = tf.image.random_brightness(final_image,max_delta=63)
        final_image = tf.image.random_contrast(final_image,lower=0.2, upper=1.8)

    # Normalize whitening
    final_image = tf.image.per_image_standardization(final_image)
    return(final_image, image_label)


In [9]:
#Create CIFAR image pipeline reader
def input_pipeline(batch_size, train_logical= True):
    if train_logical:
        files = [os.path.join(data_dir, extract_folder, 'data_batch_{}.bin'.format(i)) for i in range(1,6)]
    else:
        files = [os.path.join(data_dir, extract_folder, 'test_batch_{}.bin')]
    filename_queue = tf.train.string_input_producer(files)
    image, label = read_cifar_files(filename_queue)
    min_after_dequeue = 1000
    capacity = min_after_dequeue + 3*batch_size
    example_batch, label_batch = tf.train.shuffle_batch([image, label], batch_size, capacity, min_after_dequeue)
    return(example_batch, label_batch)

Structure of our model :
2 layer of convolutional network -> 
3 fully connected layer

In [12]:
def cifar_cnn_model(input_image, batch_size, train_logical = True):
    def truncated_normal_var(name, shape, dtype):
        return (tf.get_variable(name = name, shape = shape, dtype = dtype, 
                                initializer = tf.truncated_normal_initializer(stddev = 0.05)))
    def zero_var(name, shape, dtype):
        reuturn (tf.get_variable(name = name, shape = shape, dtype = dtype,
                                initializer = tf.constant_initializer(0.0)))
        #First Convolution layer
        with tf.variable_scope('conv1') as scope:
            #Conv_kernel  is a matrix 5*5 for 3 colors , create 64 features
            conv1_kernel = truncated_normal_var('conv1_kernel', shape = [5,5,3,64], dtype = tf.float32)
            #convolve across the image with a stride size of 1
            conv1   = tf.nn.conv2d(input_images, conv1_kernel, [1,1,1,1], padding = 'SAME')
            #Initialize and add the bias term
            conv1_bias = zeros_var(name = 'conv_bias1', shape =[64], dtype = tf.float32)
            conv1_add_bias = tf.nn.bias_add(conv1, conv1_bias)
            #ReLU element wise
            relu_conv1 = tf.nn.relu(conv1_add_bias)
            
        #Max Pooling
        pool1 = tf.nn.max_pool(relu_conv1, ksize =[1,3,3,1], strides = [1,2,2,1], padding = 'SAME', name = 'pool_layer1')
        #Local Reponse Normalization
        norm1 = tf.nn.lrn(pool1, depth_radius = 5, bias = 2.0, alpha = 1e-3, beta = 0.75, name = 'norm1')
        
        #Second Convolution Layer
        with tf.variable_scope('conv2') as scope:
            #Conv_kernel  is a matrix 5*5, across all prior 64 features , create 64 features
            conv1_kernel = truncated_normal_var('conv2_kernel', shape = [5,5,64,64], dtype = tf.float32)
            #convolve across the image with a stride size of 1
            conv2   = tf.nn.conv2d(norm1, conv2_kernel, [1,1,1,1], padding = 'SAME')
            #Initialize and add the bias term
            conv2_bias = zeros_var(name = 'conv_bias2', shape =[64], dtype = tf.float32)
            conv2_add_bias = tf.nn.bias_add(conv2, conv2_bias)
            #ReLU element wise
            relu_conv2 = tf.nn.relu(conv2_add_bias)
            
        #Max Pooling
        pool2 = tf.nn.max_pool(relu_conv2, ksize =[1,3,3,1], strides = [1,2,2,1], padding = 'SAME', name = 'pool_layer2')
        #Local Reponse Normalization
        norm2 = tf.nn.lrn(pool2, depth_radius = 5, bias = 2.0, alpha = 1e-3, beta = 0.75, name = 'norm2')
            
            